In [ ]:
import requests
import pandas as pd

API_KEY = "WUH1mjruEsf9PY4vHqIKmJjkEbI0iZil"

# Step 1: search for nearby EV chargers
search_url = "https://api.tomtom.com/search/2/nearbySearch/.json"

search_params = {
    "key": API_KEY,
    "lat": 55.6761,
    "lon": 12.5683,
    "radius": 5000,
    "categorySet": 7309,
    "limit": 20
}

search_resp = requests.get(search_url, params=search_params, timeout=30)
search_resp.raise_for_status()
search_data = search_resp.json()

# Step 2: fetch live availability for each station
availability_url = "https://api.tomtom.com/search/2/chargingAvailability.json"

rows = []

for station in search_data["results"]:
    name = station.get("poi", {}).get("name")
    address = station.get("address", {}).get("freeformAddress")
    lat = station.get("position", {}).get("lat")
    lon = station.get("position", {}).get("lon")

    charging_id = (
        station.get("dataSources", {})
        .get("chargingAvailability", {})
        .get("id")
    )

    if not charging_id:
        rows.append({
            "name": name,
            "address": address,
            "lat": lat,
            "lon": lon,
            "charging_id": None,
            "connector_type": None,
            "power_kw": None,
            "total": None,
            "available": None,
            "occupied": None,
            "reserved": None,
            "unknown": None,
            "out_of_service": None
        })
        continue

    params = {
        "key": API_KEY,
        "chargingAvailability": charging_id
    }

    try:
        avail_resp = requests.get(availability_url, params=params, timeout=30)
        avail_resp.raise_for_status()
        avail_data = avail_resp.json()

        connectors = avail_data.get("connectors", [])

        if not connectors:
            rows.append({
                "name": name,
                "address": address,
                "lat": lat,
                "lon": lon,
                "charging_id": charging_id,
                "connector_type": None,
                "power_kw": None,
                "total": None,
                "available": None,
                "occupied": None,
                "reserved": None,
                "unknown": None,
                "out_of_service": None
            })
            continue

        for c in connectors:
            connector_type = c.get("type")
            total = c.get("total")

            current = c.get("availability", {}).get("current", {})

            # Some responses also include perPowerLevel
            per_power = c.get("availability", {}).get("perPowerLevel", [])

            if per_power:
                for p in per_power:
                    rows.append({
                        "name": name,
                        "address": address,
                        "lat": lat,
                        "lon": lon,
                        "charging_id": charging_id,
                        "connector_type": connector_type,
                        "power_kw": p.get("powerKW"),
                        "total": total,
                        "available": p.get("available"),
                        "occupied": p.get("occupied"),
                        "reserved": p.get("reserved"),
                        "unknown": p.get("unknown"),
                        "out_of_service": p.get("outOfService")
                    })
            else:
                rows.append({
                    "name": name,
                    "address": address,
                    "lat": lat,
                    "lon": lon,
                    "charging_id": charging_id,
                    "connector_type": connector_type,
                    "power_kw": None,
                    "total": total,
                    "available": current.get("available"),
                    "occupied": current.get("occupied"),
                    "reserved": current.get("reserved"),
                    "unknown": current.get("unknown"),
                    "out_of_service": current.get("outOfService")
                })

    except requests.RequestException as e:
        print(f"Failed for {name}: {e}")

df = pd.DataFrame(rows)
print(df.head(20))

                                  name  \
0                           E.ON Drive   
1   Industriens Hus - Parkeringskælder   
2                               Spirii   
3                                APCOA   
4                                APCOA   
5                               Spirii   
6                         NRGi Erhverv   
7                               Norlys   
8                               Clever   
9                               Clever   
10                              Spirii   
11      E.ON Drive Infrastructure GmbH   
12                              Drivee   
13                              drivee   
14                              Clever   
15                              Clever   
16                              Clever   
17                          E.ON Drive   
18                     EDF Danmark A/S   
19                            Dyrkøb 3   

                                          address        lat        lon  \
0                   Rådhuspladsen, 1550 Kø